In [ ]:
# ===================================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# ===================================================================
# NFStream es la librería clave para analizar los archivos .pcap
# El resto son para manipulación de datos, IA y gráficos.
# El -q es para que la salida de la instalación sea más limpia (quiet).
# ===================================================================
!pip install -q nfstream pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# ===================================================================
# CELDA 2: IMPORTACIÓN DE MÓDULOS
# ===================================================================
import os
import pandas as pd
import numpy as np
import re
from datetime import datetime
from nfstream import NFStreamer
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Ignoramos advertencias para mantener la salida limpia
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")


In [ ]:
# ===================================================================
# CELDA 3: FUNCIÓN INTELIGENTE DE DESFASE
# ===================================================================
def obtener_desfase_por_csv(archivo_sync, archivo_csv):
    """
    Calcula el desfase temporal entre el timestamp del juego y el timestamp UNIX
    buscando una marca de sincronización en un archivo de log.
    """
    print(f"   -> Buscando sincronización para {archivo_csv}...")
    
    # Extrae la fecha y hora del nombre del fichero de telemetría (ej: YYYYMMDD_HHMMSS)
    match_csv = re.search(r"(\d{8}_\d{6})", archivo_csv)
    if not match_csv:
        raise ValueError("El nombre del CSV no tiene el formato de fecha esperado (YYYYMMDD_HHMMSS).")
    fecha_csv = match_csv.group(1)
    
    with open(archivo_sync, 'r') as f:
        for linea in f: # Buscar linea por linea en el archivo de sincronización
            patron = r"\[([\d\.]+)\] ==== \[SYNC_TELEMETRIA\] Tiempo interno: ([\d\.]+) ===="
            match = re.search(patron, linea) # Extrae el tiempo UNIX y el tiempo del juego
            if match:
                tiempo_unix = float(match.group(1))
                tiempo_juego_sec = float(match.group(2))
                
                # Compara si la fecha y hora del log coincide con la del archivo CSV
                fecha_unix_str = datetime.fromtimestamp(tiempo_unix).strftime("%Y%m%d_%H%M%S")
                if fecha_unix_str == fecha_csv:
                    desfase = tiempo_unix - tiempo_juego_sec
                    print(f"      [OK] Sincronización encontrada. Desfase: {desfase:.3f} s")
                    return desfase
                    
    raise ValueError(f"No se encontró en '{archivo_sync}' la fecha correspondiente al CSV '{archivo_csv}'")


In [ ]:
# ===================================================================
# CELDA 4: FUNCIÓN DE PREPROCESAMIENTO Y FUSIÓN
# ===================================================================
def preprocesar_y_fusionar(archivo_csv, pcap_local, pcap_amigo, desfase, id_host, id_amigo, tipo_red, es_ataque, fase):
    """
    Toma los datos de telemetría y red, los fusiona y enriquece para la IA.
    """
    # --- 1. PREPARAR TELEMETRÍA ---
    df_juego = pd.read_csv(archivo_csv)
    df_juego['timestamp_real'] = df_juego['timestamp'] + desfase
    df_juego = df_juego.sort_values('timestamp_real')
    print(f"   -> Telemetría leída: {len(df_juego)} filas.")

    jugadores_ids = df_juego['player_id'].unique()
    if id_host not in jugadores_ids or id_amigo not in jugadores_ids:
        print(f"      [AVISO] IDs configurados ({id_host}, {id_amigo}) no coinciden con los del CSV: {jugadores_ids}. Revisa si alguien no se conectó.")
    
    df_juego_host = df_juego[df_juego['player_id'] == id_host].copy()
    df_juego_amigo = df_juego[df_juego['player_id'] == id_amigo].copy()
    print(f"      - Datos Host (ID {id_host}): {len(df_juego_host)} filas.")
    print(f"      - Datos Amigo (ID {id_amigo}): {len(df_juego_amigo)} filas.")

    # --- Función interna para procesar PCAPs ---
    def procesar_pcap(pcap_path, tipo_conexion_str):
        print(f"   -> Procesando PCAP {tipo_conexion_str} ({pcap_path})...")
        try:
            streamer = NFStreamer(source=pcap_path, statistical_analysis=True, active_timeout=1)
            df_red = streamer.to_pandas()
            if df_red is None or df_red.empty:
                print(f"      [AVISO] El PCAP '{pcap_path}' está vacío o no contiene flujos. Se omitirá.")
                return pd.DataFrame()
            
            print(f"      - Flujos de red encontrados: {len(df_red)}.")
            df_red['tipo_conexion'] = tipo_conexion_str
            df_red['timestamp_sec'] = df_red['bidirectional_first_seen_ms'] / 1000.0
            return df_red.sort_values('timestamp_sec')
        except Exception as e:
            print(f"      [ERROR] No se pudo procesar '{pcap_path}'. Error: {e}. Se omitirá.")
            return pd.DataFrame()

    # --- 2. PROCESAR Y FUSIONAR HOST (Local) ---
    df_red_local = procesar_pcap(pcap_local, 'local')
    df_combinado_host = pd.DataFrame()
    if not df_red_local.empty and not df_juego_host.empty:
        df_combinado_host = pd.merge_asof(
            df_juego_host, df_red_local, left_on='timestamp_real', right_on='timestamp_sec', direction='nearest', tolerance=0.2
        )
        print(f"   -> Fusión Host: {len(df_combinado_host)} filas combinadas.")
    # --- 3. PROCESAR Y FUSIONAR AMIGO (Cliente) ---
    df_red_amigo = procesar_pcap(pcap_amigo, 'amigo')
    df_combinado_amigo = pd.DataFrame()
    if not df_red_amigo.empty and not df_juego_amigo.empty:
        df_combinado_amigo = pd.merge_asof(
            df_juego_amigo, df_red_amigo, left_on='timestamp_real', right_on='timestamp_sec', direction='backward', tolerance=0.2
        )#nearest daba problemas con el desfase, así que usamos backward para asegurar que tomamos el último paquete antes del evento de juego.
        print(f"   -> Fusión Amigo: {len(df_combinado_amigo)} filas combinadas.")
    else:
        # Si el PCAP del amigo falla, es un error crítico para el entrenamiento.
        raise RuntimeError(f"Fallo crítico: no se pudieron obtener datos de red para el amigo en la sesión {archivo_csv}.")

    # --- 4. JUNTAR MUNDOS ---
    df_combinado = pd.concat([df_combinado_host, df_combinado_amigo], ignore_index=True)
    df_combinado = df_combinado.ffill().fillna(0)
    
    # --- 5. INGENIERÍA DE CARACTERÍSTICAS ---
    df_combinado['delta_yaw'] = df_combinado.groupby('player_id')['yaw'].diff().abs().fillna(0)
    df_combinado['delta_pitch'] = df_combinado.groupby('player_id')['pitch'].diff().abs().fillna(0)
    df_combinado['ratio_velocidad_bytes'] = df_combinado['velocity'] / (df_combinado['bidirectional_bytes'] + 1)
    df_combinado.replace([np.inf, -np.inf], 0, inplace=True)

    # --- 6. ETIQUETADO METODOLÓGICO ---
    df_combinado['tipo_red'] = tipo_red
    df_combinado['etiqueta_real'] = -1 if es_ataque else 1
    df_combinado['fase'] = fase

    return df_combinado
